In [2]:
!pip install pyspark plotly "pandas>=2.2.0" "nbformat>=4.2.0"

In [3]:
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless
!java -version

Hit:1 https://download.docker.com/linux/ubuntu noble InRelease
Get:2 https://cli.github.com/packages stable InRelease [3917 B]                
Hit:3 https://packages.cloud.google.com/apt cloud-sdk InRelease                
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://security.ubuntu.com/ubuntu noble-security InRelease              
Hit:8 https://cloud.archive.ubuntu.com/ubuntu noble InRelease                  
Hit:9 https://archive.ubuntu.com/ubuntu noble InRelease             
Hit:10 https://cloud.archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:11 https://cloud.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:12 http://deb.wakemeops.com/wakemeops stable InRelease          
Hit:13 https://cloud.archive.ubuntu.com/ubuntu noble-security InRelease
Hit:14 https://archive.

In [4]:
import sqlite3
import os
#import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col
import os
from pyspark.sql import SparkSession


os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = SparkSession.builder \
        .master("local[4]") \
        .appName("PySpark DataFrames API") \
        .config("spark.executor.memory", "4g") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()

print("Spark Session configured and ready!")

Spark Session configured and ready!


In [5]:
#!pip install matplotlib seaborn plotly pandas

In [6]:
df = spark.read.csv('../data/monster_job_sample.csv', header=True, inferSchema=True)

In [7]:
monster= df

In [17]:
monster.show(30)

+--------------------+------------+----------+-----------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|             country|country_code|date_added|has_expired|       job_board|     job_description|           job_title|            job_type|            location|        organization|            page_url|              salary|              sector|             uniq_id|
+--------------------+------------+----------+-----------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|United States of ...|          US|      NULL|         No|jobs.monster.com|TeamSoft is seein...|IT Support Techni...|  Full Time Employee|   Madison, WI 53702|                NULL|http://jobview.mo...|    

In [18]:
from pyspark.sql import functions as F

monster.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in monster.columns
]).show()

+-------+------------+----------+-----------+---------+---------------+---------+--------+--------+------------+--------+------+------+-------+
|country|country_code|date_added|has_expired|job_board|job_description|job_title|job_type|location|organization|page_url|salary|sector|uniq_id|
+-------+------------+----------+-----------+---------+---------------+---------+--------+--------+------------+--------+------+------+-------+
|      0|           0|     21878|          0|        0|              0|        0|    1485|       8|        6495|      18| 16793|  4554|     90|
+-------+------------+----------+-----------+---------+---------------+---------+--------+--------+------------+--------+------+------+-------+



In [15]:
for col_name, dtype in df.dtypes:
    print(f"{col_name}: {dtype}")

country: string
country_code: string
date_added: string
has_expired: string
job_board: string
job_description: string
job_title: string
job_type: string
location: string
organization: string
page_url: string
salary: string
sector: string
uniq_id: string


Salary should not be string but some lines do have strings to describe the salary not just the value

In [16]:
monster.describe().show()

26/05/09 17:02:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+------------+----------+-----------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+
|summary|             country|country_code|date_added|has_expired|       job_board|     job_description|           job_title|            job_type|            location|        organization|            page_url|              salary|    sector|             uniq_id|
+-------+--------------------+------------+----------+-----------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+
|  count|               22000|       22000|       122|      22000|           22000|               22000|               22000|               20515|               21992|               15505|               21982|  